# 04b — Transformer Fine-Tuning on Google Colab (Phase 3)

Same job as notebook 04, but built to run on **Google Colab with a free T4
GPU** instead of the MacBook. Why we moved: the balanced 6-month data is bigger,
and a T4 chews through it far faster than the Mac's GPU, with no overnight wait.

**What is different from the Mac notebook 04:**
- Runs on **CUDA (T4)**, not Apple MPS.
- Reads the data from **Google Drive** (you upload the two parquet files once).
- Trains only **two configs per task**: weighted-CE vs focal, both at lr 2e-4.
  No learning-rate sweep — that saves most of the time. We already know 2e-4 is
  a good rate; the one thing that might change with more/balanced data is which
  *loss* wins, so that is the only thing we still compare.
- **Early stopping**: each run trains until the validation score stops
  improving, instead of a fixed number of epochs.
- **Checkpoints on Google Drive**: after a run finishes it is saved to Drive.
  If Colab disconnects or errors, just run the notebook again — finished runs
  are skipped and only the unfinished one repeats. (See the note in step 6.)

## One-time setup before you run
1. In Colab: **Runtime -> Change runtime type -> T4 GPU**.
2. In your Google Drive, make a folder `complaint-intelligence` and inside it a
   folder `data/processed`. Upload your two files there:
   `train.parquet` and `test.parquet` (the balanced ones that notebook 01 made).
3. Run the cells top to bottom. The first cell installs libraries (~1 min) and
   Colab may ask you to allow Google Drive access.

When it finishes, the results file and figures are written back to the same
Drive folder, ready to copy into your repo.


In [ ]:
# ------------------------------------------------------------------
# 1) Install the libraries Colab does not already have.
# ------------------------------------------------------------------
# Colab already ships PyTorch with the correct CUDA build, so we do NOT reinstall
# torch (reinstalling it can break the GPU). We only add the NLP + tracking libs.
!pip install -q "transformers>=4.40,<5" "peft>=0.10" "datasets>=2.19" "accelerate>=0.30" mlflow
print("install done")


<local path>: line 2: <local path>: No such file or directory
<local path>: line 2: exec: <local path>: cannot execute: No such file or directory
install done


In [ ]:
# ------------------------------------------------------------------
# 2) Connect Google Drive and set up the folders we read from / write to.
# ------------------------------------------------------------------
import os
from google.colab import drive
drive.mount("/content/drive")

# The project folder you made in Drive (step 2 of the setup above).
DRIVE_DIR = "/content/drive/MyDrive/complaint-intelligence"

# Where the data lives, and where we will save results, figures, and checkpoints.
DATA_DIR      = os.path.join(DRIVE_DIR, "data/processed")
REPORTS_DIR   = os.path.join(DRIVE_DIR, "reports")
FIGURES_DIR   = os.path.join(DRIVE_DIR, "reports/figures")
CKPT_DIR      = os.path.join(DRIVE_DIR, "checkpoints")   # best adapters saved here
RESULTS_DIR   = os.path.join(DRIVE_DIR, "run_results")   # one small json per run

for folder in [REPORTS_DIR, FIGURES_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)

# Quick check that the data is actually where we expect it.
train_path         = os.path.join(DATA_DIR, "train.parquet")          # balanced (not used here)
train_natural_path = os.path.join(DATA_DIR, "train_natural.parquet")  # natural mix, used below
test_path          = os.path.join(DATA_DIR, "test.parquet")
print("train_natural file found:", os.path.exists(train_natural_path))
print("test file found:         ", os.path.exists(test_path))
if not os.path.exists(train_natural_path):
    print("\n>>> Upload train_natural.parquet and test.parquet into", DATA_DIR)
    print("    (both are produced by notebook 01 - rerun it if train_natural.parquet is missing)")


In [ ]:
# ------------------------------------------------------------------
# 3) Imports, fixed random seeds, and confirm we are on the GPU.
# ------------------------------------------------------------------
import time
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# On Colab with a T4 this should print 'cuda'. If it prints 'cpu', go to
# Runtime -> Change runtime type -> T4 GPU, then run the notebook again.
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("WARNING: no GPU found - this will be very slow. Switch the runtime to T4.")

print("torch version:", torch.__version__)
print("device:", device)


In [ ]:
# ------------------------------------------------------------------
# 4) Load data. Carve a NATURAL validation set, then balance only train.
# ------------------------------------------------------------------
# Why read the natural (unbalanced) train here: we want the validation set to
# have the REAL class mix, so early-stopping is judged on the same kind of
# distribution as the test set. We carve val FIRST (so it stays natural), then
# balance only the training part - the validation set is never balanced.
train_natural = pd.read_parquet(train_natural_path)
test = pd.read_parquet(test_path)

train_natural = train_natural.sort_values("date_received").reset_index(drop=True)

# 1) Validation = newest 15% of the natural train (left at its real class mix).
cut = int(len(train_natural) * 0.85)
train_raw = train_natural.iloc[:cut].reset_index(drop=True)
val = train_natural.iloc[cut:].reset_index(drop=True)

# 2) Balance ONLY the training slice (same soft cap as notebook 01).
MAX_PER_CLASS = 8000
balanced_parts = []
for product_name in train_raw["product"].unique():
    rows = train_raw[train_raw["product"] == product_name]
    if len(rows) > MAX_PER_CLASS:
        rows = rows.sample(n=MAX_PER_CLASS, random_state=SEED)
    balanced_parts.append(rows)
train = pd.concat(balanced_parts).sort_values("date_received").reset_index(drop=True)

print("train (balanced):", len(train), " val (natural):", len(val), " test:", len(test))
print("product classes:", train["product"].nunique())
print("monetary positives  train:", int(train["is_monetary_relief"].sum()),
      " val:", int(val["is_monetary_relief"].sum()),
      " test:", int(test["is_monetary_relief"].sum()))


In [ ]:
# ------------------------------------------------------------------
# 5) Tokenization analysis: how long are the narratives in TOKENS?
# ------------------------------------------------------------------
# DistilBERT reads at most 512 tokens. We measure lengths to justify our cut-off.
# We keep the first 256 tokens (head truncation): it covers most of each
# narrative, keeps training fast, and complaints usually state the issue up front.
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_texts = train["consumer_complaint_narrative"].sample(
    n=min(4000, len(train)), random_state=SEED).tolist()

token_lengths = []
for text in sample_texts:
    ids = tokenizer.encode(text, add_special_tokens=True, truncation=False)
    token_lengths.append(len(ids))

token_lengths = np.array(token_lengths)
for p in [50, 75, 90, 95, 99]:
    print("p" + str(p) + " token length:", int(np.percentile(token_lengths, p)))
print("over 512 tokens:", round(100 * (token_lengths > 512).mean(), 1), "%")
print("over 256 tokens:", round(100 * (token_lengths > 256).mean(), 1), "%")

plt.figure(figsize=(7, 4))
plt.hist(np.clip(token_lengths, 0, 800), bins=50)
plt.axvline(256, color="red", linestyle="--", label="chosen max_length = 256")
plt.xlabel("tokens per narrative")
plt.ylabel("count")
plt.title("Narrative length in DistilBERT tokens (train sample)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "token_length_hist.png"), dpi=120)
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 6) Settings + a helper that turns text + labels into batches.
# ------------------------------------------------------------------
MAX_LENGTH = 256      # keep the first 256 tokens of each narrative
BATCH_SIZE = 32       # a T4 handles 32 comfortably at length 256
MAX_EPOCHS = 6        # upper limit; early stopping usually stops sooner
PATIENCE = 2          # stop if the validation score does not improve for 2 epochs

from torch.utils.data import TensorDataset, DataLoader

# Build the product label <-> number maps from the data itself.
product_names = sorted(train["product"].unique().tolist())
label2id = {name: i for i, name in enumerate(product_names)}
id2label = {i: name for name, i in label2id.items()}
print(len(product_names), "product classes")


def make_loader(texts, labels, shuffle):
    # Tokenize a list of texts and wrap them + their labels in a DataLoader.
    # Every narrative is padded/truncated to exactly MAX_LENGTH tokens.
    encoded = tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    label_tensor = torch.tensor(list(labels), dtype=torch.long)
    dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"], label_tensor)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)


In [ ]:
# ------------------------------------------------------------------
# 7) Helpers: build a LoRA model, class weights, evaluate, loss function.
# ------------------------------------------------------------------
from transformers import (AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score


def build_lora_model(num_labels):
    # DistilBERT + a fresh classification head, wrapped in LoRA so we only train
    # a tiny number of extra weights.
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels)
    lora = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,               # Tier 1: raised from 8 for a bit more capacity
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_lin", "v_lin"],
    )
    model = get_peft_model(base, lora)
    return model.to(device)


def load_saved_model(num_labels, adapter_dir):
    # Rebuild a model from a saved LoRA adapter (used to score the winner on TEST,
    # and to reuse a run that finished in an earlier session).
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels)
    model = PeftModel.from_pretrained(base, adapter_dir)
    return model.to(device)


def compute_class_weights(int_labels, num_classes):
    # Rare classes get a bigger weight so the model does not ignore them.
    counts = np.bincount(int_labels, minlength=num_classes)
    weights = len(int_labels) / (num_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float, device=device)


def evaluate_classifier(model, loader):
    # Run the model over a loader; return true labels, predicted labels, probs.
    model.eval()
    all_true, all_pred, all_prob = [], [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            all_true.append(labels.numpy())
            all_pred.append(preds.cpu().numpy())
            all_prob.append(probs.cpu().numpy())
    return (np.concatenate(all_true), np.concatenate(all_pred), np.concatenate(all_prob))


def loss_from_logits(logits, labels, class_weights, loss_type):
    # plain_ce   : ordinary cross-entropy (used for PRODUCT - already balanced by sampling)
    # weighted_ce: class-weighted cross-entropy (used for MONETARY - its only correction)
    # focal      : focuses on the hard examples (class_weights may be None)
    if loss_type == "plain_ce":
        return F.cross_entropy(logits, labels)
    if loss_type == "weighted_ce":
        return F.cross_entropy(logits, labels, weight=class_weights)
    if loss_type == "focal":
        ce = F.cross_entropy(logits, labels, weight=class_weights, reduction="none")
        pt = torch.exp(-ce)
        gamma = 2.0
        return (((1 - pt) ** gamma) * ce).mean()
    raise ValueError("unknown loss_type: " + str(loss_type))


## How checkpointing works here (read once)

There are **four training runs** total: product {weighted-CE, focal} and
monetary {weighted-CE, focal}. Each run trains a fresh model with early stopping.

After a run finishes, two things are written to Google Drive:
- the run's **best adapter** (the small LoRA weights) in `checkpoints/`,
- a tiny **result file** in `run_results/` recording its validation score.

When the training function starts a run, it first checks for that result file.
If it exists (the run finished in an earlier session), it **skips training and
reuses the saved adapter**. So if Colab disconnects or a run crashes, you just
run the whole notebook again: the runs that already finished are skipped, and
only the interrupted run repeats from the start.

On top of that, the best adapter is re-saved every time an epoch improves, so
even a mid-run crash leaves the best-so-far weights on Drive. (This is
per-run checkpointing, not mid-epoch resume — the runs are short enough on a T4
that repeating one interrupted run is cheap, and this keeps the code simple and
reliable. If you ever want true mid-run resume, that is a small add-on.)


In [ ]:
# ------------------------------------------------------------------
# 8) One training function, with early stopping + Drive checkpointing.
# ------------------------------------------------------------------
# 'watch' is the validation score we maximize and early-stop on:
#   'macro_f1' for the 9-class product task, 'pr_auc' for the rare monetary task.
import mlflow
mlflow.set_tracking_uri("file:" + os.path.join(DRIVE_DIR, "mlruns"))


def load_or_train(run_name, num_labels, train_loader, val_loader,
                  class_weights, loss_type, lr, watch):
    result_file = os.path.join(RESULTS_DIR, run_name + ".json")
    best_dir = os.path.join(CKPT_DIR, run_name + "_best")

    # --- Restart-safe skip: this run already finished in an earlier session. ---
    if os.path.exists(result_file):
        with open(result_file) as f:
            saved = json.load(f)
        print(run_name, "already done - val", watch, round(saved["val_score"], 4),
              "(skipping training)")
        return saved["val_score"], best_dir

    print("training", run_name, "...")
    model = build_lora_model(num_labels)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr)
    total_steps = len(train_loader) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

    best_score = -1.0
    epochs_without_improvement = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for input_ids, attention_mask, labels in train_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = loss_from_logits(logits, labels, class_weights, loss_type)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

        # Validation score for this epoch.
        val_true, val_pred, val_prob = evaluate_classifier(model, val_loader)
        if watch == "macro_f1":
            score = f1_score(val_true, val_pred, average="macro")
        else:
            score = average_precision_score(val_true, val_prob[:, 1])
        print("  epoch", epoch + 1, "/", MAX_EPOCHS, "| val", watch, round(score, 4))

        # Keep the best epoch. Save its adapter to Drive right away, so even a
        # crash leaves the best-so-far weights on disk. Stop early if stuck.
        if score > best_score:
            best_score = score
            epochs_without_improvement = 0
            model.save_pretrained(best_dir)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print("  no improvement for", PATIENCE, "epochs - stopping early")
                break

    # Record the result so a future restart skips this run.
    with open(result_file, "w") as f:
        json.dump({"run_name": run_name, "loss_type": loss_type, "lr": lr,
                   "watch": watch, "val_score": float(best_score)}, f)

    # Log the run to MLflow too (saved on Drive under mlruns/).
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("base_model", MODEL_NAME)
        mlflow.log_param("method", "LoRA")
        mlflow.log_param("loss_type", loss_type)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("max_length", MAX_LENGTH)
        mlflow.log_metric("val_" + watch, float(best_score))

    return best_score, best_dir


In [ ]:
# ------------------------------------------------------------------
# 9) Task A - product classification. Two configs: plain CE vs focal.
# ------------------------------------------------------------------
# Tier 1 change: NO class weights here. The training slice is already balanced
# by sampling, so weighting on top was correcting for imbalance twice and made
# the model over-predict the rare classes. One correction is enough.
mlflow.set_experiment("phase3_transformer_6mo")

y_train_prod = train["product"].map(label2id).tolist()
y_val_prod = val["product"].map(label2id).tolist()
train_loader_prod = make_loader(train["consumer_complaint_narrative"], y_train_prod, shuffle=True)
val_loader_prod = make_loader(val["consumer_complaint_narrative"], y_val_prod, shuffle=False)

# Config 1: plain cross-entropy (no weights).
f1_ce, dir_ce = load_or_train(
    "product_plain_ce_v2", len(product_names),
    train_loader_prod, val_loader_prod, None,
    "plain_ce", 2e-4, "macro_f1")

# Config 2: focal loss (also no weights).
f1_focal, dir_focal = load_or_train(
    "product_focal_v2", len(product_names),
    train_loader_prod, val_loader_prod, None,
    "focal", 2e-4, "macro_f1")

if f1_ce >= f1_focal:
    prod_best_loss, prod_best_dir, prod_best_val = "plain_ce", dir_ce, f1_ce
else:
    prod_best_loss, prod_best_dir, prod_best_val = "focal", dir_focal, f1_focal

print()
print("validation macro-F1:  plain_ce", round(f1_ce, 4), " focal", round(f1_focal, 4))
print("best product config:", prod_best_loss, "(val macro-F1", round(prod_best_val, 4), ")")


In [ ]:
# ------------------------------------------------------------------
# 10) Score the best product model once on TEST, compare to baseline.
# ------------------------------------------------------------------
from sklearn.metrics import accuracy_score, classification_report

# Reload the winner from Drive (works even if training was skipped this session).
best_product_model = load_saved_model(len(product_names), prod_best_dir)

test_loader_prod = make_loader(
    test["consumer_complaint_narrative"], test["product"].map(label2id).tolist(), shuffle=False)
test_true, test_pred, _ = evaluate_classifier(best_product_model, test_loader_prod)

prod_macro_f1 = f1_score(test_true, test_pred, average="macro")
prod_accuracy = accuracy_score(test_true, test_pred)
print("TEST  macro-F1:", round(prod_macro_f1, 4), "  accuracy:", round(prod_accuracy, 4))
print("Old March baseline to beat:  macro-F1 0.7316")
print("(Also compare against your NEW 6-month baseline from notebook 03.)")
print()
print(classification_report(test_true, test_pred,
                            target_names=product_names, digits=3, zero_division=0))


In [ ]:
# ------------------------------------------------------------------
# 11) Task B - monetary-relief risk (binary, ~3% positive). Two configs.
# ------------------------------------------------------------------
# Tier 1 note: we KEEP class weights here. The product-balancing above does
# nothing for the monetary label (still ~3% positive), so the class weights are
# this task's ONLY imbalance correction - dropping them would make the model
# predict 'no relief' for almost everything.
y_train_mon = train["is_monetary_relief"].astype(int).tolist()
y_val_mon = val["is_monetary_relief"].astype(int).tolist()
train_loader_mon = make_loader(train["consumer_complaint_narrative"], y_train_mon, shuffle=True)
val_loader_mon = make_loader(val["consumer_complaint_narrative"], y_val_mon, shuffle=False)
class_weights_mon = compute_class_weights(np.array(y_train_mon), 2)

mon_ce, mon_dir_ce = load_or_train(
    "monetary_weighted_ce_v2", 2,
    train_loader_mon, val_loader_mon, class_weights_mon,
    "weighted_ce", 2e-4, "pr_auc")

mon_focal, mon_dir_focal = load_or_train(
    "monetary_focal_v2", 2,
    train_loader_mon, val_loader_mon, class_weights_mon,
    "focal", 2e-4, "pr_auc")

if mon_ce >= mon_focal:
    mon_best_loss, mon_best_dir, mon_best_val = "weighted_ce", mon_dir_ce, mon_ce
else:
    mon_best_loss, mon_best_dir, mon_best_val = "focal", mon_dir_focal, mon_focal

print()
print("validation PR-AUC:  weighted_ce", round(mon_ce, 4), " focal", round(mon_focal, 4))
print("best monetary config:", mon_best_loss, "(val PR-AUC", round(mon_best_val, 4), ")")


In [ ]:
# ------------------------------------------------------------------
# 12) Score monetary on TEST + calibration + 'flag top X%' analysis.
# ------------------------------------------------------------------
from sklearn.calibration import calibration_curve

mon_best_model = load_saved_model(2, mon_best_dir)

test_loader_mon = make_loader(
    test["consumer_complaint_narrative"], test["is_monetary_relief"].astype(int).tolist(),
    shuffle=False)
test_true_mon, _, test_prob_mon = evaluate_classifier(mon_best_model, test_loader_mon)
pos_scores = test_prob_mon[:, 1]

mon_pr_auc = average_precision_score(test_true_mon, pos_scores)
mon_roc_auc = roc_auc_score(test_true_mon, pos_scores)
print("TEST  PR-AUC:", round(mon_pr_auc, 4), "  ROC-AUC:", round(mon_roc_auc, 4))
print("Old March baseline to beat:  PR-AUC 0.3138")
print()

# Calibration curve: do predicted probabilities match reality?
frac_pos, mean_pred = calibration_curve(test_true_mon, pos_scores, n_bins=10, strategy="quantile")
plt.figure(figsize=(6, 5))
plt.plot(mean_pred, frac_pos, marker="o", label="DistilBERT")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfectly calibrated")
plt.xlabel("predicted probability of monetary relief")
plt.ylabel("actual fraction that got relief")
plt.title("Calibration - monetary-relief model")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "monetary_calibration.png"), dpi=120)
plt.show()

# Business framing: if reviewers only read the riskiest X%, how many real
# monetary-relief cases do they catch?
order = np.argsort(-pos_scores)
total_pos = int(test_true_mon.sum())
print()
for pct in [0.05, 0.10, 0.20]:
    k = int(len(pos_scores) * pct)
    caught = int(test_true_mon[order[:k]].sum())
    print("review top " + str(int(pct * 100)) + "%: catch " + str(caught) + "/" +
          str(total_pos) + " = " + str(round(100 * caught / total_pos, 1)) + "% of cases")


In [ ]:
# ------------------------------------------------------------------
# 13) Inference speed: how much slower is the transformer than the baseline?
# ------------------------------------------------------------------
sample = test["consumer_complaint_narrative"].head(200).tolist()
enc = tokenizer(sample, truncation=True, padding="max_length",
                max_length=MAX_LENGTH, return_tensors="pt")
best_product_model.eval()
start = time.time()
with torch.no_grad():
    _ = best_product_model(input_ids=enc["input_ids"].to(device),
                           attention_mask=enc["attention_mask"].to(device))
elapsed = time.time() - start
ms_each = 1000 * elapsed / len(sample)
print("DistilBERT: " + str(round(ms_each, 1)) + " ms per complaint on " + str(device))
print("(TF-IDF + LogReg baseline is well under 1 ms per complaint on CPU.)")


In [ ]:
# ------------------------------------------------------------------
# 14) Write the results into reports/transformer_report.md (on Drive).
# ------------------------------------------------------------------
import datetime
lines = []
lines.append("# Phase 3 - Transformer results (DistilBERT + LoRA, 6-month balanced data)")
lines.append("")
lines.append("Generated by notebook 04b on Colab. Device: " + str(device) +
             ". max_length = " + str(MAX_LENGTH) + ", batch = " + str(BATCH_SIZE) +
             ", early stopping (max " + str(MAX_EPOCHS) + " epochs, patience " + str(PATIENCE) + ").")
lines.append("")
lines.append("## Task A - product classification (9 classes, macro-F1)")
lines.append("")
lines.append("| model | macro-F1 | accuracy |")
lines.append("|---|---|---|")
lines.append("| Old March baseline (LogReg word+char) | 0.7316 | 0.8501 |")
lines.append("| DistilBERT + LoRA (" + prod_best_loss + ") | " +
             str(round(prod_macro_f1, 4)) + " | " + str(round(prod_accuracy, 4)) + " |")
lines.append("")
lines.append("Best product config on validation: " + prod_best_loss +
             " (val macro-F1 " + str(round(prod_best_val, 4)) + "). " +
             "weighted_ce val " + str(round(f1_ce, 4)) + " vs focal val " + str(round(f1_focal, 4)) + ".")
lines.append("")
lines.append("## Task B - monetary relief (binary, ~3% positive)")
lines.append("")
lines.append("| model | PR-AUC | ROC-AUC |")
lines.append("|---|---|---|")
lines.append("| Old March baseline (LogReg word) | 0.3138 | 0.9324 |")
lines.append("| DistilBERT + LoRA (" + mon_best_loss + ") | " +
             str(round(mon_pr_auc, 4)) + " | " + str(round(mon_roc_auc, 4)) + " |")
lines.append("")
lines.append("Best monetary config on validation: " + mon_best_loss +
             " (val PR-AUC " + str(round(mon_best_val, 4)) + "). " +
             "weighted_ce val " + str(round(mon_ce, 4)) + " vs focal val " + str(round(mon_focal, 4)) + ".")
lines.append("")
lines.append("Inference: " + str(round(ms_each, 1)) + " ms per complaint on " +
             str(device) + " (baseline < 1 ms on CPU).")
lines.append("")
lines.append("NOTE: compare against your NEW 6-month baseline (notebook 03), not only")
lines.append("the old March numbers above, for a fair same-data comparison.")
lines.append("")

report_path = os.path.join(REPORTS_DIR, "transformer_report.md")
with open(report_path, "w") as f:
    f.write("\n".join(lines))
print("wrote", report_path)
print()
print("\n".join(lines))


In [ ]:
# ------------------------------------------------------------------
# 15) Bundle everything and download it to your computer in one file.
# ------------------------------------------------------------------
# Packs the two trained adapters, the report, the figures, and the MLflow logs
# into a single zip and downloads it straight to your Mac's Downloads folder.
# The markdown cell below says where to unzip each piece in your repo.
import shutil
from google.colab import files

bundle_dir = "/content/phase3_outputs"
if os.path.exists(bundle_dir):
    shutil.rmtree(bundle_dir)
os.makedirs(bundle_dir)

shutil.copytree(prod_best_dir, os.path.join(bundle_dir, "models/product_distilbert_lora"))
shutil.copytree(mon_best_dir, os.path.join(bundle_dir, "models/monetary_distilbert_lora"))
shutil.copy(report_path, os.path.join(bundle_dir, "transformer_report.md"))
shutil.copytree(FIGURES_DIR, os.path.join(bundle_dir, "figures"))
colab_mlruns = os.path.join(DRIVE_DIR, "mlruns")
if os.path.exists(colab_mlruns):
    shutil.copytree(colab_mlruns, os.path.join(bundle_dir, "mlruns_colab"))

shutil.make_archive("/content/phase3_outputs", "zip", bundle_dir)
print("zipped - your browser should now download phase3_outputs.zip")
files.download("/content/phase3_outputs.zip")


## After the run - unzip into your repo

The cell above downloads `phase3_outputs.zip` to your Mac. Unzip it and place
each piece into your local repo:

| from the zip | into your repo |
|---|---|
| `models/product_distilbert_lora/` | `models/product_distilbert_lora/` |
| `models/monetary_distilbert_lora/` | `models/monetary_distilbert_lora/` |
| `transformer_report.md` | `reports/transformer_report.md` |
| `figures/*.png` | `reports/figures/` |
| `mlruns_colab/` | `mlruns_colab/` (view separately; don't merge into local `mlruns/`) |

### Loading a saved model later (error analysis or serving)

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

base = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=9)   # 9 for product, 2 for monetary
model = PeftModel.from_pretrained(base, "models/product_distilbert_lora")
model.eval()
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
```

Then paste the four headline numbers into `reports/model_card.md`.
